# Analyze Recorded DJ Set to Produce Chromagram Vectors

This notebook produces chromagram vectors, per time step, for the given recorded show.

Each chromagram vector covers the entire piano keyboard, per note. Stated more technically, each chromagram vector covers each note within the scientific pitch notation range C0 through B8. 

## Import useful libraries

In [1]:
import librosa
import pandas as pd

In [2]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F

In [3]:
from chromagram_functions import process_octave

## User settings

In [4]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 100
spark_memory = '70G'

# location of the recorded DJ set which we want to analyze
filename_show = '/home/emily/Desktop/projects/dj/song_recognition/data/partial-goth-set.mp3'

## Load recorded show and extract harmonic content

In [5]:
y_show, sr_show = librosa.load(filename_show)
y_show_harmonic, y_show_percussive = librosa.effects.hpss(y_show)

## Define function for extracting per time step features of the recorded DJ set

In [6]:
def process_show(y_harmonic, sampling_rate, hop_length):
    y_tick_labels = librosa.key_to_notes('C:major')  # we compute this every time for now
    results_list = []
    for octave_number in range(0, 9):
        chromagram = process_octave(y_harmonic, octave_number, sr = sampling_rate, hop_length = hop_length)
        df = pd.DataFrame(chromagram.T)
        df.columns = [x + str(octave_number) for x in y_tick_labels]
        results_list.append(df)
    df_all_octaves = pd.concat(results_list, axis = 1)
    return df_all_octaves

## Compute DJ set features per time step

In [7]:
pdf_show = process_show(y_show_harmonic, sampling_rate, hop_length)

In [8]:
pdf_show.head(3)

,C0,C♯0,D0,D♯0,E0,F0,F♯0,G0,G♯0,A0,...,D8,D♯8,E8,F8,F♯8,G8,G♯8,A8,A♯8,B8
0,0.295346,0.151470,0.097726,0.252986,0.138511,0.243578,0.212136,0.285559,0.312221,0.367504,...,0.123332,0.052991,0.109756,0.252345,0.402005,0.209243,0.001852,0.116505,0.249286,0.777105
1,0.294280,0.155600,0.095750,0.249192,0.139231,0.240703,0.213539,0.279795,0.306799,0.370414,...,0.121413,0.051049,0.105946,0.251179,0.423754,0.210312,0.004149,0.115905,0.245322,0.767880
2,0.293380,0.159106,0.093833,0.244912,0.139621,0.237798,0.214577,0.274433,0.301529,0.373136,...,0.121002,0.050304,0.102868,0.250164,0.444329,0.211568,0.007364,0.116227,0.241185,0.757766


## QA

In [9]:
len(pdf_show.index)

6504

In [10]:
len(pdf_show.dropna().index)

6504

## Identify the column names for the notes

In [11]:
pitch_columns = [x for x in pdf_show.columns if x != 'id']

## Enumerate the time steps

In [12]:
pdf_show['time_step'] = pdf_show.index

## Initialize a Spark session

In [13]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

26/03/19 17:40:50 WARN Utils: Your hostname, emily-MS-7B96 resolves to a loopback address: 127.0.1.1; using 192.168.1.99 instead (on interface eno1)
26/03/19 17:40:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 17:40:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/19 17:40:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Convert Pandas DF to Spark DF and collapse pitch columns to a vector column

In [14]:
sdf_show = (
    spark
    .createDataFrame(pdf_show)
    .orderBy('time_step')
    .withColumn('array_show', F.array(*pitch_columns))
    .select('time_step', 'array_show')
)

In [15]:
sdf_show.show(5)

26/03/19 17:40:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------------------+
|time_step|          array_show|
+---------+--------------------+
|        0|[0.29534608125686...|
|        1|[0.29428040981292...|
|        2|[0.29337969422340...|
|        3|[0.29257827997207...|
|        4|[0.29183560609817...|
+---------+--------------------+
only showing top 5 rows



## Save for later

In [16]:
path_show_output = output_directory + '/show_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_show.write.mode('overwrite').parquet(path_show_output)

## Close Spark session

In [17]:
spark.stop()